# 自作モジュール：平均を求める関数を別のコードから使う

対応する本文は [自作モジュール](../chapters/modules.tex) の章末課題である。まず平均の関数を実行し、次にファイルへ保存して別のPythonから呼び出す。最後に、インストール方法によってソースの変更がどう反映されるかを比較する。

## 問2の準備：関数を動かす

1、2、3の平均は2である。次のセルでは関数を定義してから呼び出す。`return` は計算した値を呼び出し元へ返す。空のリストでは平均を計算できないので、`ValueError` を出す。

In [1]:
def mean_signal(values):
    """数値列の算術平均を返す。空列ではValueErrorを出す。"""
    if len(values) == 0:
        raise ValueError("入力列が空です。")
    return sum(values) / len(values)

print(mean_signal([1.0, 2.0, 3.0]))

2.0


表示は2.0となる。同じ関数を別の解析からも使えるように、次にファイルへ保存する。

## 問1：ファイルの置き場所を決める

本文の `uv init --lib signal_tools` は、他のコードから読み込んで使うライブラリの構成を作る。作成するファイルと役割は次のとおりである。

```text
signal_tools/
├── pyproject.toml
├── src/
│   └── signal_tools/
│       ├── __init__.py
│       ├── statistics.py
│       └── py.typed
└── tests/
    └── test_statistics.py
```

`statistics.py` に平均の関数を書く。`__init__.py` は、使う側が `from signal_tools import mean_signal` と書けるようにする。`tests/test_statistics.py` には、正しい結果と例外を確かめるコードを書く。`src/` は、このパッケージのコードを置くフォルダーである。

### 保存先を用意する

この後のセルではファイルを作る。次のセルを実行して、`notebooks/_outputs/11_modules/` を用意する。表示された場所に結果が保存され、同名の実習用ファイルは再実行すると置き換わる。

Notebookは教材の `notebooks` フォルダー、または教材の一番上のフォルダーから実行する。`Path.cwd()` は現在のフォルダーを調べ、`mkdir()` は保存先を作る。

In [2]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "main.tex").is_file():
    raise FileNotFoundError("教材のフォルダーか notebooks フォルダーから実行してください。")
WORK = ROOT / "notebooks" / "_outputs" / "11_modules"
WORK.mkdir(parents=True, exist_ok=True)
print("Python", sys.version.split()[0])
print("保存先:", WORK.relative_to(ROOT))

Python 3.12.14
保存先: notebooks/_outputs/11_modules


### 設定と公開する関数名を書く

次のセルはフォルダーを作り、`pyproject.toml` と `__init__.py` を保存する。`name` は配布するときの名前、`requires-python` は対応するPythonの範囲である。NumPyは使う側でも必要なライブラリとして記録する。pytestとruffは開発時に使う道具として記録する。

ファイルを組み立ててインストールする仕組みをビルドバックエンドという。本文のuvはuv_buildを用いる。ここでは、導入済みのsetuptoolsを使って同じ平均関数のインストールを比較する。バックエンドごとに設定は異なる。

In [3]:
import subprocess
import textwrap

PROJECT = WORK / "signal_tools"
PACKAGE = PROJECT / "src" / "signal_tools"
PACKAGE.mkdir(parents=True, exist_ok=True)
(PROJECT / "tests").mkdir(exist_ok=True)
(PROJECT / "pyproject.toml").write_text(textwrap.dedent('''[build-system]
requires = ["setuptools>=64", "wheel"]
build-backend = "setuptools.build_meta"

[project]
name = "signal-tools"
version = "0.1.0"
requires-python = ">=3.10"
dependencies = ["numpy>=2"]

[dependency-groups]
dev = ["pytest", "ruff"]

[tool.setuptools.packages.find]
where = ["src"]
'''), encoding="utf-8")
(PACKAGE / "__init__.py").write_text("from .statistics import mean_signal\n\n__all__ = ['mean_signal']\n", encoding="utf-8")
(PACKAGE / "py.typed").touch()
print((PROJECT / "pyproject.toml").read_text())

[build-system]
requires = ["setuptools>=64", "wheel"]
build-backend = "setuptools.build_meta"

[project]
name = "signal-tools"
version = "0.1.0"
requires-python = ">=3.10"
dependencies = ["numpy>=2"]

[dependency-groups]
dev = ["pytest", "ruff"]

[tool.setuptools.packages.find]
where = ["src"]



## 問2と問3：関数とテストをファイルへ保存する

次のセルは、最初に試した平均関数へ型ヒントと詳しいdocstringを加え、`statistics.py` に保存する。`Sequence[float]` は数値の列を受け取る想定、`-> float` は数値を返す想定を表す。型ヒントだけでは実行時の型検査は行われない。

テストは2件ある。一つは平均が2になる入力、もう一つは空入力で例外が出る場合である。テスト内の `assert` は条件が違うと止まり、`pytest.raises()` は指定した例外が出ることを確かめる。

In [4]:
source = textwrap.dedent('''from collections.abc import Sequence


def mean_signal(values: Sequence[float]) -> float:
    """信号列の算術平均を返す。

    Args:
        values: 数値の列。
    Returns:
        入力値の算術平均。
    Raises:
        ValueError: 入力列が空の場合。
    """
    if len(values) == 0:
        raise ValueError("入力列が空です。")
    return sum(values) / len(values)
''')
(PACKAGE / "statistics.py").write_text(source, encoding="utf-8")
(PROJECT / "tests" / "test_statistics.py").write_text(textwrap.dedent('''import pytest
from signal_tools import mean_signal


def test_mean_signal():
    assert mean_signal([1.0, 2.0, 3.0]) == pytest.approx(2.0)


def test_empty_input():
    with pytest.raises(ValueError):
        mean_signal([])
'''), encoding="utf-8")

225

保存した関数の内容を表示する。`Args` の下が受け取る値、`Returns` の下が返す値、`Raises` の下が例外の条件である。

In [5]:
print((PACKAGE / "statistics.py").read_text(encoding="utf-8"))
print((PROJECT / "tests" / "test_statistics.py").read_text(encoding="utf-8"))

from collections.abc import Sequence


def mean_signal(values: Sequence[float]) -> float:
    """信号列の算術平均を返す。

    Args:
        values: 数値の列。
    Returns:
        入力値の算術平均。
    Raises:
        ValueError: 入力列が空の場合。
    """
    if len(values) == 0:
        raise ValueError("入力列が空です。")
    return sum(values) / len(values)

import pytest
from signal_tools import mean_signal


def test_mean_signal():
    assert mean_signal([1.0, 2.0, 3.0]) == pytest.approx(2.0)


def test_empty_input():
    with pytest.raises(ValueError):
        mean_signal([])



## 問4：インストール先を二つ用意する

通常インストールでは、関数をインストール先へコピーする。editable installでは、作業中のソースファイルを参照する。違いを比べるため、別々の仮想環境を作る。

端末での基本形は、通常が `uv pip install .`、editableが `uv pip install -e .` である。下のセルはその比較を自動で準備する。`--no-index --no-deps --no-build-isolation` は、導入済みのビルド・検証ツールを使い、追加のダウンロードを行わずに実習するための指定である。通常の依存関係の導入では、この3指定をそのまま付ける必要はない。

In [6]:
# Notebook 用 venv の site-packages にあるビルド・検証ツールを共有する。
import site
import venv

shared_site = next(p for p in site.getsitepackages() if Path(p).is_dir() and "site-packages" in p)
environments = {}
for name, editable in [("regular", False), ("editable", True)]:
    env_dir = WORK / name
    venv.EnvBuilder(with_pip=True).create(env_dir)
    executable = env_dir / ("Scripts/python.exe" if sys.platform == "win32" else "bin/python")
    site_dir = subprocess.run([str(executable), "-c", "import site; print(site.getsitepackages()[0])"],
                              capture_output=True, text=True, check=True).stdout.strip()
    (Path(site_dir) / "textbook_dependencies.pth").write_text(shared_site + "\n", encoding="utf-8")
    command = [str(executable), "-m", "pip", "install", "--no-index", "--no-deps", "--no-build-isolation", "--quiet"]
    if editable:
        command.append("-e")
    subprocess.run(command + [str(PROJECT)], cwd=WORK, capture_output=True, text=True, check=True)
    environments[name] = executable

### どちらからも同じ関数を呼ぶ

それぞれのPythonを新しく起動して関数を呼ぶ。最初は両方が2.0を返す。既に読み込んだ関数のキャッシュと混同しないよう、毎回別のプロセスで起動する。

In [7]:
for name, executable in environments.items():
    result = subprocess.run([str(executable), "-c", "from signal_tools import mean_signal; print(mean_signal([1,2,3]))"],
                            cwd=WORK, capture_output=True, text=True, check=True)
    assert float(result.stdout) == 2.0
    print(name, result.stdout.strip())

regular 2.0
editable 2.0


### 保存したテストと説明文を確認する

次のセルでpytestを実行する。末尾に `2 passed` と出れば、通常入力と空入力の両方のテストに通っている。続く `help()` の表示では、先ほど保存した説明文を読む。

In [8]:
tested = subprocess.run([str(environments["editable"]), "-m", "pytest", "-q", str(PROJECT / "tests")],
                        cwd=WORK, capture_output=True, text=True, check=True)
print(tested.stdout)
assert "2 passed" in tested.stdout

..                                                                       [100%]
2 passed in 0.00s



In [9]:
help_result = subprocess.run([str(environments["editable"]), "-c", "from signal_tools import mean_signal; help(mean_signal)"],
                             cwd=WORK, capture_output=True, text=True, check=True)
print(help_result.stdout)

Help on function mean_signal in module signal_tools.statistics:

mean_signal(values: collections.abc.Sequence[float]) -> float
    信号列の算術平均を返す。

    Args:
        values: 数値の列。
    Returns:
        入力値の算術平均。
    Raises:
        ValueError: 入力列が空の場合。




### ソースを変更して違いを見る

作業中の関数が返す値を一時的に-99へ変える。通常インストールでは2.0のまま、editableでは-99.0になることを確認する。確認後は `finally` の処理で元のソースへ戻す。

In [10]:
# 関数の動作を変え、再起動したプロセスで差を確認した後、元へ戻す。
try:
    changed = source.replace("return sum(values) / len(values)", "return -99.0")
    (PACKAGE / "statistics.py").write_text(changed, encoding="utf-8")
    observed = {}
    for name, executable in environments.items():
        result = subprocess.run([str(executable), "-c", "from signal_tools import mean_signal; print(mean_signal([1,2,3]))"],
                                cwd=WORK, capture_output=True, text=True, check=True)
        observed[name] = float(result.stdout)
    assert observed == {"regular": 2.0, "editable": -99.0}
    print(observed)
finally:
    (PACKAGE / "statistics.py").write_text(source, encoding="utf-8")

{'regular': 2.0, 'editable': -99.0}


## 問5：実行時と開発時の依存関係を分ける

uv を使う場合は `uv add numpy` と `uv add --dev pytest ruff` を実行する。上の pyproject.toml でも同じ分類を確認できる。`.venv`、キャッシュ、配布生成物は再生成できるため Git から除く。`uv.lock` は共同開発の環境をそろえるためコミットする方針が考えられる。一方、利用者への配布依存関係は pyproject.toml で定義する。

通常の開発ではパッケージをインストールした環境で `uv run pytest` と `uv run ruff check .` を実行する。

In [11]:
(PROJECT / ".gitignore").write_text(".venv/\n__pycache__/\n.pytest_cache/\ndist/\n", encoding="utf-8")
result = subprocess.run([sys.executable, "-m", "ruff", "check", str(PROJECT / "src"), str(PROJECT / "tests")],
                        capture_output=True, text=True, check=True)
print(result.stdout)
print((PROJECT / ".gitignore").read_text())

All checks passed!

.venv/
__pycache__/
.pytest_cache/
dist/

